# 00 · Setup and data
Downloads the 6 MVTec AD categories **straight into Google Drive**, so nothing touches your laptop.
Runtime: CPU is fine (Runtime → Change runtime type → CPU).

### Setup (run every session)
Mounts Google Drive (where data, caches and results live), clones or updates the repo, installs requirements.
**Edit `REPO_URL` once** to point at your GitHub repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/<your-username>/defectlens.git"   # <-- change this
import os
if not os.path.exists('/content/defectlens'):
    !git clone -q {REPO_URL} /content/defectlens
else:
    !git -C /content/defectlens pull -q
%cd /content/defectlens
!pip install -q -r requirements.txt
!mkdir -p /content/drive/MyDrive/defectlens/work

## Option A — official MVTec AD download (recommended)
1. Open https://www.mvtec.com/company/research/datasets/mvtec-ad, accept the license (CC BY-NC-SA 4.0) and get the download link(s).
2. Paste them below. Either the full archive link or the individual category links work; only our 6 categories are kept.

In [ ]:
MVTEC_URLS = [
    # "https://...paste link here...",
]
CATEGORIES = ["carpet", "leather", "tile", "bottle", "screw", "metal_nut"]
DATA_ROOT = "/content/drive/MyDrive/defectlens/data/mvtec_ad"

import os, glob, shutil, subprocess
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs("/content/mvtec_tmp", exist_ok=True)
for i, url in enumerate(MVTEC_URLS):
    archive = f"/content/mvtec_tmp/part{i}.tar.xz"
    subprocess.run(["wget", "-q", "-O", archive, url], check=True)
    subprocess.run(["tar", "-xf", archive, "-C", "/content/mvtec_tmp"], check=True)
    os.remove(archive)   # free Colab disk as we go

## Option B — Kaggle mirror (alternative)
Needs a Kaggle account: upload your `kaggle.json` API token when prompted and set `KAGGLE_SLUG` to the MVTec AD dataset you choose on kaggle.com (check that it includes `ground_truth/` masks).

In [ ]:
KAGGLE_SLUG = ""   # e.g. "<owner>/<dataset-name>" copied from the dataset page
if KAGGLE_SLUG:
    from google.colab import files
    files.upload()   # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d {KAGGLE_SLUG} -p /content/mvtec_tmp --unzip -q

## Keep only our 6 categories and copy them to Drive

In [ ]:
for cat in CATEGORIES:
    hits = [p for p in glob.glob(f"/content/mvtec_tmp/**/{cat}", recursive=True)
            if os.path.isdir(os.path.join(p, "train"))]
    if not hits:
        print("missing:", cat); continue
    dst = os.path.join(DATA_ROOT, cat)
    if not os.path.exists(dst):
        shutil.copytree(hits[0], dst)
    n_train = len(glob.glob(f"{dst}/train/good/*.png"))
    n_test = len(glob.glob(f"{dst}/test/*/*.png"))
    n_mask = len(glob.glob(f"{dst}/ground_truth/*/*.png"))
    print(f"{cat:10s} train {n_train:4d}  test {n_test:4d}  masks {n_mask:4d}")